# EkaQuant Evaluation Notebook
This notebook evaluates **Qwen2.5-3B-Instruct** using the **eka-eval** framework and prepares the environment for **EkaQuant** task-aware selective quantization experiments.

In [13]:
from kaggle_secrets import UserSecretsClient

# 1. Hugging Face Authentication
# Ensure you have a Kaggle Secret named 'HF_TOKEN'
user_secrets = UserSecretsClient()
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        from huggingface_hub import login

        login(token=hf_token)
    else:
        print("Warning: HF_TOKEN not found in Kaggle Secrets.")
except Exception as e:
    print(f"Could not authenticate with HF_TOKEN: {e}")

In [14]:
# 2. Install Dependencies
!pip install -q transformers bitsandbytes accelerate peft 
!pip install -q numpy scipy kneed scikit-image tqdm

# 3. Install eka-eval from IIT Gandhinagar
!git clone https://github.com/lingo-iitgn/eka-eval.git
%cd eka-eval
!pip install -q evaluate rouge_score datasets
!pip install -e .
%cd ..

# 4. Install EkaQuant
!git clone https://github.com/TarunNagarajan/EkaQuant.git
%cd EkaQuant
!pip install -e .
%cd ..

fatal: destination path 'eka-eval' already exists and is not an empty directory.
/kaggle/working/eka-eval
Obtaining file:///kaggle/working/eka-eval
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for eka-eval (pyproject.toml) ... done
  Created wheel for eka-eval: filename=eka_eval-0.1.0-0.editable-py3-none-any.whl size=10869 sha256=026f52f2dcda78cf9da955ecb0052586412f224df3bb8047b25294d755d56976
  Stored in directory: /tmp/pip-ephem-wheel-cache-le8izq3o/wheels/bc/03/2f/c2b2705dd2205b1317e087a2d5213eef67c7ea4bf6100fa5db
Successfully built eka-eval
  Attempting uninstall: eka-eval
    Found existing installation: eka-eval 0.1.0
    Uninstalling eka-eval-0.1.0:
      Successfully uninstalled eka-eval-0.1.0
/kaggle/working
fatal: destination path 'EkaQuant' already exists and is not an empty directory.
/kaggle/w

In [15]:
# 5. Run Evaluation Baselines
model_id = "Qwen/Qwen2.5-3B-Instruct"
model_name = model_id.split("/")[-1]

# 8-bit Evaluation
print(f"Starting 8-bit evaluation for {model_id}...")
# eka-eval hardcodes 4bit in its model_loader.py; we patch it to 8bit for this run
!sed -i 's/load_in_4bit *= *True/load_in_8bit=True/g' eka-eval/eka_eval/core/model_loader.py
# eka-eval has a typo in its config for MMLU-IN; we patch the module path
!sed -i 's/indic\.mmlu_in\.evaluate_mmlu_in/multilingual.mmlu_in.evaluate_mmlu_in/g' eka-eval/eka_eval/config/benchmark_config.py
# Pipe responses to handle interactive prompts: [Local, HF, Model ID, No custom BM, Group 9 (INDIC), Task 1 (MMLU-IN), No visualization]
!printf "1\n1\n{model_id}\nno\n9\n1\nno\n" | python eka-eval/scripts/run_benchmarks.py --results_dir results_8bit_{model_name}

# 4-bit Evaluation
print(f"Starting 4-bit evaluation for {model_id}...")
# Reverting the patch back to 4bit
!sed -i 's/load_in_8bit *= *True/load_in_4bit=True/g' eka-eval/eka_eval/core/model_loader.py
!printf "1\n1\n{model_id}\nno\n9\n1\nno\n" | python eka-eval/scripts/run_benchmarks.py --results_dir results_4bit_{model_name}

Starting 8-bit evaluation for Qwen/Qwen2.5-3B-Instruct...
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2212, in __getattr__
    module = self._get_module(self._class_to_module[name])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2446, in _get_module
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2444, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstr

In [16]:
# 6. Summary of Results
import glob
import pandas as pd

csv_files = glob.glob("./results_*/calculated.csv", recursive=True)
if not csv_files:
    print("No results found yet.")
for f in csv_files:
    print(f"\n--- {f} ---")
    df = pd.read_csv(f)
    display(df)

No results found yet.
